# Data Parsing for BSSE Project

This notebook parses the existing data in the `data` directory.

The goal of this notebook is to read the data into data structures that are conducive to analysis. At the same time, we want to build in sanity checks to help us ensure that calculations ran successfully and that calculations are what we think they are. The end goal is to have tables of data extracted from each calculation, such that rows index calculations and the columns index properties (e.g., geometry, basis, energy components).

We refer to calculations based on the cluster they pertain to. We use the following shorthand for naming calculations:

- `{system}-{real}-{basis}-{parameters}` where `{system}` is one or more underscore-separated monomers and `{real}` and `{basis}` are one or more underscore-separated monomer indices.
    - `Ne_Ne-1_2-1_2-{parameters}` is a neon dimer in the dimer basis set.
    - `Ne_Ne-1-1_2-{parameters}` computes the energy of the first Ne atom using the dimer basis set.
    - `Ne_Ne_Ne-1-1_3-{parameters}` computes the energy of the first Ne in the dimer basis formed from monomers 1 and 3.

Misc. Notes.

- Data for the project lives in `bsse_db/data/monomer_name`, where "monomer_name" is the molecular formula of the monomers in the cluster.
- We are going to assume that for a monomer containing $n$ atoms, the first $n$ atoms in a file belong to monomer 1, the next $n$ belong to monomer 2, the next $n$ belong to monomer 3, etc.
- Ghost atoms in NWChem are specified by prepending `bq` to the atomic symbol.
- Initial data allowed the molecular system to be reoriented. This leads to small geometric differences between the supersystem and subsystem geometries.

In [43]:
import tarfile
import itertools
import os
import math
import pandas as pd
from nwchem_helpers.parse_nwchem_output import parse_nwchem_output
from nwchem_helpers.compare_results import similar_nwchem_runs

def parse_tarred_file(tarball, file_path):
    f = tarball.extractfile(file_path)
    content = f.read().decode("utf-8").split('\n')
    return parse_nwchem_output(iter(content))

def save_state(key, new_results, collected_results):
    if key in collected_results:
        assert similar_nwchem_runs(collected_results[key], new_results)
    else:
        collected_results[key] = new_results

def make_key_from_file(monomers, file_name):
    key_base = '_'.join(monomers)
    real_piece = None
    basis_piece = None

    w_numbers = file_name.replace('A', '1').replace('B', '2').replace('C', '3')
    
    if 'output_E_' in w_numbers:
        # Filename format is "basis" then "real"
        system = w_numbers.split('output_E_')[1].split('.txt')[0]
        basis_piece, real_piece = system.split('_')
    
    elif w_numbers.count('_') == 2:
        # Filename format is "real" then "basis"
        system = w_numbers.split('output_')[1].split('.txt')[0]
        real_piece, basis_piece = system.split('_')
    elif file_name in ['output_dimer.txt']:
        real_piece = '12'
        basis_piece = '12'
    elif file_name in ['output_monomer.txt']:
        real_piece = '1'
        basis_piece = '1'
    else:
            raise Exception('Unrecognized filename')
    
    real_piece ='_'.join(list(real_piece))
    basis_piece = '_'.join(list(basis_piece))
    
    return key_base + '-' + real_piece + '-' + basis_piece

## Ne Clusters

- `Ne_dimers.tar` contain CCSD(T)/aug-cc-pvdz calculations.
- `Ne_Ne_distance_x_y_z` directory contains a dimer where one of the monomers has been translated by $\vec{r} = (x,y,z)^T$.
- Translating like this duplicates effort because of the system's symmetry (i.e., only the distance between the Neons matters)
- No 'output_E_B_B.txt' because monomers are the same.

In [41]:
nsteps    = 7    # The total number of displacements along each axis
step_size = 0.25 # How much we displace for each step.

ne_dimer_distances = set()
ne_dimer_results = {}

def compute_ne_distance(geom):
    carts = [[float(geom[i][j]) for j in range(1, 4)] for i in range(2)]
    return math.dist(carts[0], carts[1])

with tarfile.open('data/Ne/Ne_dimers.tar', 'r') as tarball:
    all_names = tarball.getnames() # Gets all the directories and files inside the tarball
    
    for dx, dy, dz in itertools.product(range(1, nsteps), range(1, nsteps), range(1, nsteps)):
        x, y, z = (dx * step_size, dy * step_size, dz * step_size)
        directory_name = os.path.join('Ne_dimers', 'Ne_Ne_distance_{}_{}_{}'.format(x, y, z))

        monomers = ('Ne', 'Ne')
        if directory_name in all_names: # Checks translation is in tarball
            
            # Extract the results from the dimer file
            dimer_file_name = 'output_E_AB_AB.txt'
            file_name = os.path.join(directory_name, dimer_file_name)
            results = parse_tarred_file(tarball, file_name)
            
            # Compute and record the separation distance
            geom = results['Input Geometry (angstroms)']
            r = compute_ne_distance(geom)
            ne_dimer_distances.add(r)

            # Save the state
            key = make_key_from_file(monomers, dimer_file_name) + '-{}'.format(r)
            save_state(key, results, ne_dimer_results)

            for monomer_file in ['output_E_AB_A.txt', 'output_E_AB_B.txt']:
                file_name = os.path.join(directory_name, monomer_file)
                results = parse_tarred_file(tarball, file_name)
    
                # Sanity check it's the same distance
                geom = results['Input Geometry (angstroms)']
                r_new = compute_ne_distance(geom)
                assert math.isclose(r, r_new, abs_tol=1E-7) # NWChem only prints about 8 decimal places

                # Save the state (use dimer distance for consistency)
                key = make_key_from_file(monomers, monomer_file) + '-{}'.format(r)
                save_state(key, results, ne_dimer_results)

            for monomer_file in ['output_E_A_A.txt']:
                file_name = os.path.join(directory_name, monomer_file)
                results = parse_tarred_file(tarball, file_name)
                key = make_key_from_file(monomers, monomer_file)
                save_state(key, results, ne_dimer_results)
                
pd.DataFrame(ne_dimer_results).transpose()

,Input Geometry (angstroms),AO Basis Set,Total SCF Energy (a.u.),Total MP2 Energy (a.u.),Total CCSD Energy (a.u.),Total CCSD(T) Energy (a.u.)
Ne_Ne-1_2-1_2-1.5411035,"[(Ne, 0.00000000, 0.00000000, -0.77055175), (N...",aug-cc-pvdz,-256.874113911105,-257.292842423520881,-257.299248528910653,-257.305108143114182
Ne_Ne-1-1_2-1.5411035,"[(Ne, 0.00000000, 0.00000000, 0.00000000), (bq...",aug-cc-pvdz,-128.496953904925,-128.706906455561153,-128.710024487065823,-128.712905546045221
Ne_Ne-2-1_2-1.5411035,"[(bqNe, 0.00000000, 0.00000000, -1.54110350), ...",aug-cc-pvdz,-128.496953904925,-128.706906455561125,-128.710024474655171,-128.712905531504191
Ne_Ne-1-1,"[(Ne, 0.00000000, 0.00000000, 0.00000000)]",aug-cc-pvdz,-128.496349730514,-128.705409599288487,-128.708487837634010,-128.711294152566950
Ne_Ne-1_2-1_2-1.60078106,"[(Ne, 0.00000000, 0.00000000, -0.80039053), (N...",aug-cc-pvdz,-256.902970297102,-257.321600541022690,-257.328049289253840,-257.333887071280117
...,...,...,...,...,...,...
Ne_Ne-1-1_2-2.46221446,"[(Ne, 0.00000000, 0.00000000, 0.00000000), (bq...",aug-cc-pvdz,-128.496439816931,-128.705570759155108,-128.708655551024947,-128.711479578227227
Ne_Ne-2-1_2-2.46221446,"[(bqNe, 0.00000000, 0.00000000, -2.46221445), ...",aug-cc-pvdz,-128.496439816931,-128.705570759155023,-128.708655541533858,-128.711479570102568
Ne_Ne-1_2-1_2-2.59807622,"[(Ne, 0.00000000, 0.00000000, -1.29903811), (N...",aug-cc-pvdz,-256.991935285700,-257.410324413582885,-257.416530286745228,-257.422199193337235
Ne_Ne-1-1_2-2.59807622,"[(Ne, 0.00000000, 0.00000000, 0.00000000), (bq...",aug-cc-pvdz,-128.496428160688,-128.705549214647675,-128.708635670441282,-128.711457349988592


- `Ne_pvtz_dimers.tar` contains dimers computed using the CCSD(T)/aug-cc-pvtz model.
- The directories are now labeled `Ne_dimer_x_pvtz` where `x` is the separation distance
- The output file names have changed so that the basis set now comes second (when the basis is the same, it is just labeled dimer or monomer)

In [44]:
nsteps = 6
step_size = 0.5 # How much we displace for each step.

ne_dimer_atz_distances = set()
ne_dimer_atz_results = {}

with tarfile.open('data/Ne/Ne_pvtz_dimers.tar', 'r') as tarball:
    all_names = tarball.getnames() # Gets all the directories and files inside the tarball

    for dx in range(1, nsteps):
        x = 1.0 + dx * step_size
        directory_name = os.path.join('.', 'Ne_dimer_{}_pvtz'.format(x))

        monomers = ('Ne', 'Ne')
        if directory_name in all_names: # Checks translation is in tarball
            
            # Extract the results from the dimer file
            dimer_file_name = 'output_dimer.txt'
            file_name = os.path.join(directory_name, dimer_file_name)
            results = parse_tarred_file(tarball, file_name)

            # Compute and record the separation distance
            geom = results['Input Geometry (angstroms)']
            r = compute_ne_distance(geom)
            ne_dimer_atz_distances.add(r)

            # Save the state
            key = make_key_from_file(monomers, dimer_file_name) + '-{}'.format(r)
            save_state(key, results, ne_dimer_atz_results)

            for monomer_file in ['output_A_AB.txt', 'output_B_AB.txt']:
                file_name = os.path.join(directory_name, monomer_file)
                results = parse_tarred_file(tarball, file_name)
    
                # Sanity check it's the same distance
                geom = results['Input Geometry (angstroms)']
                r_new = compute_ne_distance(geom)
                assert math.isclose(r, r_new, abs_tol=1E-7) # NWChem only prints about 8 decimal places

                # Save the state (use dimer distance for consistency)
                key = make_key_from_file(monomers, monomer_file) + '-{}'.format(r)
                save_state(key, results, ne_dimer_atz_results)

            for monomer_file in ['output_monomer.txt']:
                file_name = os.path.join(directory_name, monomer_file)
                results = parse_tarred_file(tarball, file_name)
                key = make_key_from_file(monomers, monomer_file)
                save_state(key, results, ne_dimer_atz_results)
                
pd.DataFrame(ne_dimer_atz_results).transpose()

,Input Geometry (angstroms),AO Basis Set,Total SCF Energy (a.u.),Total MP2 Energy (a.u.),Total CCSD Energy (a.u.),Total CCSD(T) Energy (a.u.)
Ne_Ne-1_2-1_2-1.5,"[(Ne, 0.00000000, 0.00000000, -0.75000000), (N...",aug-cc-pvtz,-256.923672845821,-257.500169812455624,-257.503218017563540,-257.514180012615839
Ne_Ne-1-1_2-1.5,"[(Ne, 0.00000000, 0.00000000, 0.00000000), (bq...",aug-cc-pvtz,-128.533531563097,-128.820651191251017,-128.821907466585003,-128.827284211274645
Ne_Ne-2-1_2-1.5,"[(bqNe, 0.00000000, 0.00000000, -1.50000000), ...",aug-cc-pvtz,-128.533531563097,-128.820651191251244,-128.821907443522917,-128.827284185765222
Ne_Ne-1-1,"[(Ne, 0.00000000, 0.00000000, 0.00000000)]",aug-cc-pvtz,-128.533272825173,-128.819179150492886,-128.820401339676209,-128.825757401386966
Ne_Ne-1_2-1_2-2.0,"[(Ne, 0.00000000, 0.00000000, -1.00000000), (N...",aug-cc-pvtz,-257.052355933878,-257.625537282064158,-257.628326651500458,-257.639154692444492
Ne_Ne-1-1_2-2.0,"[(Ne, 0.00000000, 0.00000000, 0.00000000), (bq...",aug-cc-pvtz,-128.533415917756,-128.819679384212975,-128.820923628219589,-128.826284924889450
Ne_Ne-2-1_2-2.0,"[(bqNe, 0.00000000, 0.00000000, -2.00000000), ...",aug-cc-pvtz,-128.533415917756,-128.819679384212947,-128.820923628962788,-128.826284925708620
Ne_Ne-1_2-1_2-2.5,"[(Ne, 0.00000000, 0.00000000, -1.25000000), (N...",aug-cc-pvtz,-257.065201699975,-257.637601444777886,-257.640146813667513,-257.650918195551810
Ne_Ne-1-1_2-2.5,"[(Ne, 0.00000000, 0.00000000, 0.00000000), (bq...",aug-cc-pvtz,-128.533316407679,-128.819340955135885,-128.820572282893153,-128.825930816576715
Ne_Ne-2-1_2-2.5,"[(bqNe, 0.00000000, 0.00000000, -2.50000000), ...",aug-cc-pvtz,-128.533316407679,-128.819340955136028,-128.820572276969529,-128.825930810336700


- `Ne_trimers.tar` contain CCSD(T)/aug-cc-pvdz calculations.
- `Ne_trimer_x` directory contains a trimer arranged in an equilateral triangle with sides of length $x$.
- Side lengths range from 1.5 to 4 in 0.25 increments.

In [55]:
nsteps = 12
step_size = 0.25 # How much we displace for each step.

ne_trimer_distances = set()
ne_trimer_results = {}

def powerset(s): 
    return itertools.chain.from_iterable(itertools.combinations(s, r) for r in range(1, len(s) + 1))

def compute_ne_triangle_distance(geom):
    carts = [[float(geom[i][j]) for j in range(1, 4)] for i in range(3)]
    dr01 = math.dist(carts[0], carts[1])
    dr02 = math.dist(carts[0], carts[2])
    dr12 = math.dist(carts[1], carts[2])

    assert math.isclose(dr01, dr02, abs_tol=1e-7)
    assert math.isclose(dr01, dr12, abs_tol=1e-7)
    assert math.isclose(dr01, dr02, abs_tol=1e-7)

    return dr01

with tarfile.open('data/Ne/Ne_trimers.tar', 'r') as tarball:
    all_names = tarball.getnames() # Gets all the directories and files inside the tarball

    monomers = ('Ne', 'Ne', 'Ne')
    for dx in range(nsteps):
        x = 1.25 + dx * step_size
        directory_name = os.path.join('Ne_trimers', 'Ne_trimer_{}'.format(x))

        if directory_name in all_names: #             
            for real in powerset(['A', 'B', 'C']):
            
                trimer_file_name = 'output_E_ABC_{}.txt'.format(''.join(real))
                file_name = os.path.join(directory_name, trimer_file_name)
                results = parse_tarred_file(tarball, file_name)

                # Validate equilateral triangle
                r = compute_ne_triangle_distance(results['Input Geometry (angstroms)'])
                assert math.isclose(x, r, abs_tol=1E-7)
                ne_trimer_distances.add(x)

                key = make_key_from_file(monomers, trimer_file_name) + '-{}'.format(x)
                save_state(key, results, ne_trimer_results)

            for basis in itertools.combinations(['A', 'B', 'C'], 2):
                for real in powerset(basis):
                    dimer_file_name = 'output_E_{}_{}.txt'.format(''.join(basis), ''.join(real))
                    file_name = os.path.join(directory_name, dimer_file_name)
                    results = parse_tarred_file(tarball, file_name)
                    key = make_key_from_file(monomers, dimer_file_name) + '-{}'.format(x)
                    save_state(key, results, ne_trimer_results)

            for real in ['A', 'B', 'C']:
                monomer_file_name = 'output_E_{}_{}.txt'.format(''.join(real), ''.join(real))
                file_name = os.path.join(directory_name, monomer_file_name)
                results = parse_tarred_file(tarball, file_name)
                key = make_key_from_file(monomers, monomer_file_name) + '-{}'.format(x)
                save_state(key, results, ne_trimer_results)

pd.DataFrame(ne_trimer_results).transpose()

,Input Geometry (angstroms),AO Basis Set,Total SCF Energy (a.u.),Total MP2 Energy (a.u.),Total CCSD Energy (a.u.),Total CCSD(T) Energy (a.u.)
Ne_Ne_Ne-1-1_2_3-1.5,"[(Ne, 0.00000000, 0.00000000, 0.00000000), (bq...",aug-cc-pvdz,-128.497551531891,-128.708596548693720,-128.711749123182017,-128.714705781201559
Ne_Ne_Ne-2-1_2_3-1.5,"[(bqNe, 0.00000000, -0.75000000, -1.29903811),...",aug-cc-pvdz,-128.497551531891,-128.708596548693720,-128.711749176773310,-128.714705837894599
Ne_Ne_Ne-3-1_2_3-1.5,"[(bqNe, 0.00000000, -0.75000000, 1.29903811), ...",aug-cc-pvdz,-128.497551531891,-128.708596548693890,-128.711749123182187,-128.714705781201729
Ne_Ne_Ne-1_2-1_2_3-1.5,"[(Ne, -0.75000000, 0.00000000, 0.00000000), (N...",aug-cc-pvdz,-256.850480956150,-257.271543723237755,-257.278032739823232,-257.284065727582515
Ne_Ne_Ne-1_3-1_2_3-1.5,"[(Ne, 0.75000000, 0.00000000, 0.00000000), (bq...",aug-cc-pvdz,-256.850480956154,-257.271543723242303,-257.278032877622536,-257.284065870572817
...,...,...,...,...,...,...
Ne_Ne_Ne-3-2_3-3.75,"[(bqNe, 0.00000000, 0.00000000, 3.75000000), (...",aug-cc-pvdz,-128.496353557400,-128.705446233468109,-128.708526823803709,-128.711339262749618
Ne_Ne_Ne-2_3-2_3-3.75,"[(Ne, 0.00000000, 0.00000000, 1.87500000), (Ne...",aug-cc-pvdz,-256.992702929935,-257.410923282951103,-257.417085803999669,-257.422714127045253
Ne_Ne_Ne-1-1-3.75,"[(Ne, 0.00000000, 0.00000000, 0.00000000)]",aug-cc-pvdz,-128.496349730514,-128.705409599288345,-128.708487837651006,-128.711294152490353
Ne_Ne_Ne-2-2-3.75,"[(Ne, 0.00000000, 0.00000000, 0.00000000)]",aug-cc-pvdz,-128.496349730514,-128.705409599288345,-128.708487837650921,-128.711294152490240


# FH  Clusters

- Note the files refer to them as HF monomers, but we prefer FH to avoid confusion with Hartree-Fock
- File naming for dimers follows the same convention as Ne dimers.
- Directories are accidentally called `FF_distance_x_y_z` instead of `HF_HF_distance_x_y_z` (or maybe it represents the F-F distance?)


In [59]:
nsteps    = 7    # The total number of displacements along each axis
step_size = 0.25 # How much we displace for each step.

fh_dimer_displacements = set()
fh_dimer_results = {}

with tarfile.open('data/HF/HF_dimers.tar', 'r') as tarball:
    all_names = tarball.getnames() # Gets all the directories and files inside the tarball
    
    for dx, dy, dz in itertools.product(range(1, nsteps), range(1, nsteps), range(1, nsteps)):
        x, y, z = (dx * step_size, dy * step_size, dz * step_size)
        directory_name = os.path.join('HF_dimers', 'FF_distance_{}_{}_{}'.format(x, y, z))

        monomers = ('HF', 'HF')
        if directory_name in all_names: # Checks translation is in tarball
            
            # Extract the results from the dimer file
            dimer_file_name = 'output_E_AB_AB.txt'
            file_name = os.path.join(directory_name, dimer_file_name)
            results = parse_tarred_file(tarball, file_name)
            
            # Compute and record the separation distance
            geom = results['Input Geometry (angstroms)']
            fh_dimer_displacements.add((x, y, z))

            # Save the state
            key = make_key_from_file(monomers, dimer_file_name) + '-{}_{}_{}'.format(x, y, z)
            save_state(key, results, fh_dimer_results)

            for monomer_file in ['output_E_AB_A.txt', 'output_E_AB_B.txt']:
                file_name = os.path.join(directory_name, monomer_file)
                results = parse_tarred_file(tarball, file_name)
                key = make_key_from_file(monomers, monomer_file) + '-{}_{}_{}'.format(x, y, z)
                save_state(key, results, fh_dimer_results)

            for monomer_file in ['output_E_A_A.txt']:
                file_name = os.path.join(directory_name, monomer_file)
                results = parse_tarred_file(tarball, file_name)
                key = make_key_from_file(monomers, monomer_file)
                save_state(key, results, fh_dimer_results)
                
pd.DataFrame(fh_dimer_results).transpose()

,Input Geometry (angstroms),AO Basis Set,Total SCF Energy (a.u.),Total MP2 Energy (a.u.),Total CCSD Energy (a.u.),Total CCSD(T) Energy (a.u.)
HF_HF-1_2-1_2-0.25_0.25_1.5,"[(H, 0.16953681, -1.67474872, 0.00000000), (F,...",aug-cc-pvdz,-199.919303096802,-200.382408268976491,-200.386733779164018,-200.396468815361175
HF_HF-1-1_2-0.25_0.25_1.5,"[(H, 0.12264125, -0.82250695, 0.00000000), (F,...",aug-cc-pvdz,-100.033655980956,-100.259915187133060,-100.263357730394688,-100.267679081856002
HF_HF-2-1_2-0.25_0.25_1.5,"[(bqH, 0.06993437, -2.54015317, 0.00000000), (...",aug-cc-pvdz,-100.033655980926,-100.259915187103289,-100.263357710315489,-100.267679063419266
HF_HF-1-1,"[(H, 0.00000000, 0.00000000, -0.83160000), (F,...",aug-cc-pvdz,-100.033125845866,-100.258094461015162,-100.261530800873004,-100.265718820634774
HF_HF-1_2-1_2-0.25_0.5_1.5,"[(H, 0.26184107, -1.67685427, 0.00000000), (F,...",aug-cc-pvdz,-199.948775552868,-200.410150590929931,-200.415008047348920,-200.424590359304233
...,...,...,...,...,...,...
HF_HF-1-1_2-1.5_1.5_1.25,"[(H, 0.60162912, -0.57410884, 0.00000000), (F,...",aug-cc-pvdz,-100.033233580039,-100.258365282475850,-100.261797649487150,-100.266018725905752
HF_HF-2-1_2-1.5_1.5_1.25,"[(bqH, 0.42986400, -3.07437569, 0.00000000), (...",aug-cc-pvdz,-100.033233580047,-100.258365282573351,-100.261797678648151,-100.266018757296408
HF_HF-1_2-1_2-1.5_1.5_1.5,"[(H, 0.68079518, -1.86112706, 0.00000000), (F,...",aug-cc-pvdz,-200.062352490193,-200.513425747198824,-200.520282870853379,-200.528845310470217
HF_HF-1-1_2-1.5_1.5_1.5,"[(H, 0.56724430, -0.60810563, 0.00000000), (F,...",aug-cc-pvdz,-100.033227195567,-100.258319139483291,-100.261752024388443,-100.265964688415551


- The `HF_pvtz_dimers.tar` tarball only contains three geometries, so we're going to skip using those.
- The internals of the tarball are similar to `Ne_pvtz_dimers.tar`